In [2]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import re
import random
from pathlib import Path

import pandas as pd
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import to_pil_image

# Paths
input_dir = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/breast_lesion_ultrasound/BrEaST-Lesions_USG-images_and_masks")
excel_path = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/breast_lesion_ultrasound/BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx")
output_dir = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/preprocessed/BrEaST-Lesions_USG-images_and_masks")
csv_path = output_dir / "captions.csv"

output_dir.mkdir(parents=True, exist_ok=True)


# Image preprocessing
transform = T.Compose([
    T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    T.Resize((256, 256)),
    T.ToTensor(),
])

# Prompt templates
templates_meta = [
    "Ultrasound of {Region} in {PatientInfo}. {Findings}. Assessment: {Condition}.",
    "Sonographic findings in {Region}: {Findings}. {PatientInfo}. Diagnosis: {Condition}.",
    "An ultrasound image of {Region} consistent with {Condition}. {Findings}.",
    "{Region} evaluated by sonography. {Findings}. Conclusion: {Condition}.",
    "{PatientInfo} underwent ultrasound imaging. {Findings} identified in {Region}, suggestive of {Condition}.",
    "Sonography: {Region}, {PatientInfo}. {Findings}. {Condition}.",
    "{PatientInfo}. Ultrasound of {Region} reveals {Findings}, consistent with {Condition}.",
    "{Condition} pattern on ultrasound. {Region} demonstrates {Findings}. {PatientInfo}.",
    "Sonographic examination of {Region}. {Findings}. {Condition}.",
    "Ultrasound performed on {PatientInfo}. Examination of {Region} revealed {Findings}. Impression: {Condition}.",
]

# Helpers
def clean_value(x, default="unknown"):
    if pd.isna(x):
        return default
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}:
        return default
    return x

def build_patient_info(row):
    age = clean_value(row.get("Age"), None)
    if age is None or age == "unknown":
        return "a patient"
    return f"a patient aged {age}"

def build_region(row):
    tissue = "breast"
    tissue_comp = clean_value(row.get("Tissue_composition"), None)
    if tissue_comp is None or tissue_comp == "unknown":
        return tissue
    return f"{tissue} with {tissue_comp} tissue composition"

def build_findings(row):
    finding_parts = []

    for col, label in [
        ("Shape", "shape"),
        ("Margin", "margin"),
        ("Echogenicity", "echogenicity"),
        ("Posterior_features", "posterior features"),
        ("Calcifications", "calcifications"),
        ("Skin_thickening", "skin thickening"),
        ("Halo", "halo"),
        ("Signs", "signs"),
        ("Symptoms", "symptoms"),
    ]:
        val = clean_value(row.get(col), None)
        if val is not None and val != "unknown":
            finding_parts.append(f"{label}: {val}")

    if not finding_parts:
        return "unremarkable findings"
    return "; ".join(finding_parts)

def build_condition(row):
    condition_parts = []

    for col in ["Diagnosis", "Classification", "Interpretation", "BIRADS", "Verification"]:
        val = clean_value(row.get(col), None)
        if val is not None and val != "unknown":
            if col == "BIRADS":
                condition_parts.append(f"BI-RADS {val}")
            else:
                condition_parts.append(val)

    if not condition_parts:
        return "unspecified condition"

    # remove duplicates while preserving order
    seen = set()
    unique_parts = []
    for part in condition_parts:
        if part not in seen:
            unique_parts.append(part)
            seen.add(part)

    return ", ".join(unique_parts)

def generate_caption(row, image_name):
    patient_info = build_patient_info(row)
    region = build_region(row)
    findings = build_findings(row)
    condition = build_condition(row)

    # deterministic template choice by image name
    idx = hash(image_name) % len(templates_meta)
    template = templates_meta[idx]

    return template.format(
        PatientInfo=patient_info,
        Region=region,
        Findings=findings,
        Condition=condition
    )

def is_real_image_file(path: Path):
    # keep case001.png, exclude case001_tumor.png and similar masks
    name = path.name.lower()
    if path.suffix.lower() not in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"]:
        return False
    if "_tumor" in name or "_mask" in name or "mask" in name:
        return False
    return bool(re.fullmatch(r"case\d+\.png", name))

# Load metadata
df = pd.read_excel(excel_path, sheet_name="BrEaST-Lesions-USG clinical dat")

# standardize filename column
df["Image_filename"] = df["Image_filename"].astype(str).str.strip()

# build a lookup from filename -> row
meta_by_filename = {
    row["Image_filename"]: row
    for _, row in df.iterrows()
}

# Process images
rows = []

image_files = sorted([p for p in input_dir.iterdir() if p.is_file() and is_real_image_file(p)])

for img_path in image_files:
    image_name = img_path.name

    if image_name not in meta_by_filename:
        print(f"Skipping {image_name}: not found in Excel metadata")
        continue

    try:
        # preprocess and save image
        img = Image.open(img_path)
        img_tensor = transform(img)
        save_path = output_dir / image_name
        to_pil_image(img_tensor).save(save_path)

        # caption
        row = meta_by_filename[image_name]
        caption = generate_caption(row, image_name)

        rows.append({
            "image_name": image_name,
            "text_caption": caption
        })

        print(f"Processed: {image_name}")

    except Exception as e:
        print(f"Failed on {image_name}: {e}")

# Save CSV
out_df = pd.DataFrame(rows)
out_df.to_csv(csv_path, index=False)

print(f"\nDone. Processed {len(rows)} images.")
print(f"Preprocessed images saved to: {output_dir}")
print(f"CSV saved to: {csv_path}")

Processed: case001.png
Processed: case002.png
Processed: case003.png
Processed: case004.png
Processed: case005.png
Processed: case006.png
Processed: case007.png
Processed: case008.png
Processed: case009.png
Processed: case010.png
Processed: case011.png
Processed: case012.png
Processed: case013.png
Processed: case014.png
Processed: case015.png
Processed: case016.png
Processed: case017.png
Processed: case018.png
Processed: case019.png
Processed: case020.png
Processed: case021.png
Processed: case022.png
Processed: case023.png
Processed: case024.png
Processed: case025.png
Processed: case026.png
Processed: case027.png
Processed: case028.png
Processed: case029.png
Processed: case030.png
Processed: case031.png
Processed: case032.png
Processed: case033.png
Processed: case034.png
Processed: case035.png
Processed: case036.png
Processed: case037.png
Processed: case038.png
Processed: case039.png
Processed: case040.png
Processed: case041.png
Processed: case042.png
Processed: case043.png
Processed: 

In [4]:
import re
import hashlib
from pathlib import Path

import pandas as pd

# Paths
input_dir = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/preprocessed/BrEaST-Lesions_USG-images_and_masks")
excel_path = Path("/Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/breast_lesion_ultrasound/BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx")
csv_path = input_dir / "captions.csv"

# =========================
# Prompt templates
# =========================
templates_meta = [
    "Ultrasound of {Region} in {PatientInfo}. {Findings}. Assessment: {Condition}.",
    "Sonographic findings in {Region}: {Findings}. {PatientInfo}. Diagnosis: {Condition}.",
    "An ultrasound image of {Region} consistent with {Condition}. {Findings}.",
    "{Region} evaluated by sonography. {Findings}. Conclusion: {Condition}.",
    "{PatientInfo} underwent ultrasound imaging. {Findings} identified in {Region}, suggestive of {Condition}.",
    "Sonography: {Region}, {PatientInfo}. {Findings}. {Condition}.",
    "{PatientInfo}. Ultrasound of {Region} reveals {Findings}, consistent with {Condition}.",
    "{Condition} pattern on ultrasound. {Region} demonstrates {Findings}. {PatientInfo}.",
    "Sonographic examination of {Region}. {Findings}. {Condition}.",
    "Ultrasound performed on {PatientInfo}. Examination of {Region} revealed {Findings}. Impression: {Condition}.",
]

# =========================
# Helpers (unchanged)
# =========================
def clean_value(x, default="unknown"):
    if pd.isna(x):
        return default
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}:
        return default
    return x

def build_patient_info(row):
    age = clean_value(row.get("Age"), None)
    if age is None or age == "unknown":
        return "a patient"
    return f"a patient aged {age}"

def build_region(row):
    tissue = "breast"
    tissue_comp = clean_value(row.get("Tissue_composition"), None)
    if tissue_comp is None or tissue_comp == "unknown":
        return tissue
    return f"{tissue} with {tissue_comp} tissue composition"

def build_findings(row):
    finding_parts = []

    for col, label in [
        ("Shape", "shape"),
        ("Margin", "margin"),
        ("Echogenicity", "echogenicity"),
        ("Posterior_features", "posterior features"),
        ("Calcifications", "calcifications"),
        ("Skin_thickening", "skin thickening"),
        ("Halo", "halo"),
        ("Signs", "signs"),
        ("Symptoms", "symptoms"),
    ]:
        val = clean_value(row.get(col), None)
        if val is not None and val != "unknown":
            finding_parts.append(f"{label}: {val}")

    if not finding_parts:
        return "unremarkable findings"
    return "; ".join(finding_parts)

def build_condition(row):
    condition_parts = []

    for col in ["Diagnosis", "Classification", "Interpretation", "BIRADS", "Verification"]:
        val = clean_value(row.get(col), None)
        if val is not None and val != "unknown":
            if col == "BIRADS":
                condition_parts.append(f"BI-RADS {val}")
            else:
                condition_parts.append(val)

    if not condition_parts:
        return "unspecified condition"

    seen = set()
    unique_parts = []
    for part in condition_parts:
        if part not in seen:
            unique_parts.append(part)
            seen.add(part)

    return ", ".join(unique_parts)

# ✅ FIXED deterministic caption
def generate_caption(row, image_name):
    patient_info = build_patient_info(row)
    region = build_region(row)
    findings = build_findings(row)
    condition = build_condition(row)

    h = int(hashlib.md5(image_name.encode()).hexdigest(), 16)
    idx = h % len(templates_meta)
    template = templates_meta[idx]

    return template.format(
        PatientInfo=patient_info,
        Region=region,
        Findings=findings,
        Condition=condition
    )

def is_real_image_file(path: Path):
    name = path.name.lower()
    if path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
        return False
    if "_tumor" in name or "_mask" in name:
        return False
    return bool(re.fullmatch(r"case\d+\.png", name))

# =========================
# Load metadata
# =========================
df = pd.read_excel(excel_path, sheet_name="BrEaST-Lesions-USG clinical dat")
df["Image_filename"] = df["Image_filename"].astype(str).str.strip()

meta_by_filename = {
    row["Image_filename"]: row
    for _, row in df.iterrows()
}

# =========================
# Build CSV ONLY
# =========================
rows = []

image_files = sorted([p for p in input_dir.iterdir() if p.is_file() and is_real_image_file(p)])

for img_path in image_files:
    image_name = img_path.name

    if image_name not in meta_by_filename:
        print(f"Skipping {image_name}: not in metadata")
        continue

    row_meta = meta_by_filename[image_name]
    caption = generate_caption(row_meta, image_name)

    # ✅ define label (you MUST decide this mapping)
    label = clean_value(row_meta.get("Classification"), "unknown").lower()

    rows.append({
        "image_path": image_name,
        "text_caption": caption,
        "label": label   # ✅ NEW COLUMN
    })

# Save CSV
out_df = pd.DataFrame(rows)
out_df.to_csv(csv_path, index=False)

print(f"\nDone. Processed {len(rows)} images.")
print(f"CSV saved to: {csv_path}")


Done. Processed 248 images.
CSV saved to: /Users/guest/Coding/Harvard/AIM2/project/BMI702_project/dataset/preprocessed/BrEaST-Lesions_USG-images_and_masks/captions.csv
